In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

print("✅ Librerías cargadas correctamente.")

In [ ]:
import os
import numpy as np
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

ruta_base = r"C:\Users\Anxo\Desktop\Proyecto final 4\archive (3)\Squat_Data\Squat_Data"
X = []
y = []
secuencia_temporal = []

print("🚀 Iniciando Preprocesamiento Avanzado...")

for root, dirs, files in os.walk(ruta_base):
    archivos_ordenados = sorted([f for f in files if f.endswith(".npy")])
    
    for archivo in archivos_ordenados:
        ruta_archivo = os.path.join(root, archivo)
        try:
            res = np.load(ruta_archivo)
            if res.shape == (132,):
                # --- 1. NORMALIZACIÓN BIOMECÁNICA ---
                # Usamos los puntos 23 y 24 (caderas en MediaPipe) para centrar
                # Los landmarks 23 y 24 están en las posiciones [23*4 : 23*4+3] y [24*4 : 24*4+3]
                hip_l = res[23*4 : 23*4+3]
                hip_r = res[24*4 : 24*4+3]
                centro_cadera = (hip_l + hip_r) / 2
                
                # Restamos el centro de la cadera a todos los puntos (X, Y, Z)
                res_norm = res.copy()
                for i in range(33): # 33 landmarks
                    res_norm[i*4 : i*4+3] = res_norm[i*4 : i*4+3] - centro_cadera
                
                secuencia_temporal.append(res_norm)
            
            if len(secuencia_temporal) == 30:
                # Guardamos la secuencia normalizada original
                X.append(secuencia_temporal)
                y.append(1 if "Valid" in root else 0)
                
                # --- 2. DATA AUGMENTATION (ESPEJO) ---
                # Invertimos el eje X (multiplicando por -1 las posiciones 0, 4, 8...)
                secuencia_espejo = np.array(secuencia_temporal).copy()
                secuencia_espejo[:, 0::4] *= -1 
                
                X.append(secuencia_espejo)
                y.append(1 if "Valid" in root else 0)
                
                secuencia_temporal = [] 
        except:
            continue

X = np.array(X)
y = to_categorical(np.array(y), num_classes=2)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ ¡Dataset duplicado y normalizado!")
print(f"Nuevas muestras totales: {len(X)} (Antes eran 715)")
print(f"Forma de X_train: {X_train.shape}")

In [ ]:
# Definimos el modelo secuencial con capas LSTM para series temporales
model = keras.Sequential([
    keras.Input(shape=(30, 132)), # 30 frames, 132 coordenadas cada uno
    layers.LSTM(64, return_sequences=True, activation='relu'),
    layers.LSTM(128, return_sequences=False, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2), # Para evitar el sobreajuste (overfitting)
    layers.Dense(32, activation='relu'),
    layers.Dense(2, activation='softmax') # Salida: [Prob_Invalida, Prob_Valida]
])

model.compile(optimizer='Adam', 
              loss='categorical_crossentropy', 
              metrics=['categorical_accuracy'])

model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 1. Definimos los "callbacks" (ayudantes del entrenamiento)
# Si el modelo no mejora en 15 épocas, se detiene solo para evitar overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# Si el aprendizaje se estanca, bajamos la velocidad (learning rate) para ser más precisos
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001)

print("🔥 Iniciando entrenamiento optimizado...")

historia = model.fit(
    X_train, 
    y_train, 
    epochs=100, 
    batch_size=32, 
    validation_data=(X_test, y_test),
    callbacks=[early_stopping, reduce_lr] # <-- Añadimos los ayudantes aquí
)

model.save('validador_sentadillas_v2_PRO.h5')

In [ ]:
import matplotlib.pyplot as plt

# Configuramos el estilo
plt.figure(figsize=(14, 5))

# 1. Gráfica de Precisión (Accuracy)
plt.subplot(1, 2, 1)
plt.plot(historia.history['categorical_accuracy'], label='Entrenamiento', color='blue', linewidth=2)
plt.plot(historia.history['val_categorical_accuracy'], label='Validación (Test)', color='orange', linewidth=2)
plt.title('Evolución de la Precisión')
plt.xlabel('Épocas')
plt.ylabel('Exactitud (0.0 - 1.0)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# 2. Gráfica de Pérdida (Loss)
plt.subplot(1, 2, 2)
plt.plot(historia.history['loss'], label='Entrenamiento', color='blue', linewidth=2)
plt.plot(historia.history['val_loss'], label='Validación (Test)', color='orange', linewidth=2)
plt.title('Evolución del Error (Loss)')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.show()

In [1]:
import numpy as np
from tensorflow.keras.models import load_model

# 1. Cargamos el modelo que acabamos de guardar
# Cambia esta línea en tu celda de predicción:
modelo_cargado = load_model('validador_sentadillas_v2_PRO.h5')

# 2. Elige un archivo .npy de tu carpeta para probar (Copia una ruta real de tu PC)
ruta_prueba = r"C:\Users\Anxo\Desktop\Proyecto final 4\archive (3)\Squat_Data\Squat_Data\Valid\0\0.npy" 

try:
    # Cargamos el archivo (recordando que tu modelo espera una secuencia de 30 frames)
    # Aquí cargamos el primer frame como ejemplo, pero lo ideal es pasarle una secuencia completa
    datos_prueba = np.load(ruta_prueba)
    
    # Si es un solo frame (132,), lo "simulamos" 30 veces para que el modelo no de error
    if datos_prueba.shape == (132,):
        datos_prueba = np.array([datos_prueba] * 30)
    
    # Ajustamos la forma para el modelo: (1, 30, 132) -> (Batch, Frames, Features)
    datos_preparados = np.expand_dims(datos_prueba, axis=0)

    # 3. Realizamos la predicción
    prediccion = modelo_cargado.predict(datos_preparados)
    clase_resultado = np.argmax(prediccion) # Coge el índice con mayor probabilidad

    # 4. Mostramos el resultado
    etiquetas = ["❌ INVALID (NULA)", "✅ VALID (VÁLIDA)"]
    print(f"\n--- RESULTADO DEL ANÁLISIS ---")
    print(f"Predicción: {etiquetas[clase_resultado]}")
    print(f"Confianza: {prediccion[0][clase_resultado] * 100:.2f}%")

except Exception as e:
    print(f"Error al probar el archivo: {e}")

KeyboardInterrupt: 

In [8]:
import os
import numpy as np
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models

# 1. CARGA Y FILTRADO (Convertimos 33 puntos a los 17 de YOLO)
ruta_base = r"C:\Users\Anxo\Desktop\Proyecto final 4\archive (3)\Squat_Data\Squat_Data"
X_yolo = []
y_yolo = []

# Mapeo: Estos son los índices de MediaPipe que coinciden con los 17 de YOLO
indices_yolo = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]

for root, dirs, files in os.walk(ruta_base):
    archivos = sorted([f for f in files if f.endswith(".npy")])
    secuencia = []
    for f in archivos:
        res = np.load(os.path.join(root, f))
        if res.shape == (132,):
            # Extraemos solo los 17 puntos que usa YOLO (x, y, z)
            puntos_reducidos = []
            for idx in indices_yolo:
                # Cogemos x, y, z de cada punto
                puntos_reducidos.extend(res[idx*4 : idx*4+3])
            secuencia.append(puntos_reducidos)
            
            if len(secuencia) == 30:
                X_yolo.append(secuencia)
                y_yolo.append(1 if "Valid" in root else 0)
                secuencia = []

X_yolo = np.array(X_yolo)
y_yolo = to_categorical(np.array(y_yolo), 2)

X_train, X_test, y_train, y_test = train_test_split(X_yolo, y_yolo, test_size=0.2)

# 2. MODELO AJUSTADO A 17 PUNTOS (51 valores de entrada)
model_yolo = models.Sequential([
    layers.Input(shape=(30, 51)), 
    layers.LSTM(64, return_sequences=True, activation='relu'),
    layers.LSTM(128, return_sequences=False, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(2, activation='softmax')
])

model_yolo.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_yolo.fit(X_train, y_train, epochs=50, validation_data=(X_test, y_test), batch_size=32)

model_yolo.save('modelo_squat_yolo.h5')
print("✅ Modelo re-entrenado para YOLO listo.")

Epoch 1/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.5011 - loss: 0.6972 - val_accuracy: 0.5798 - val_loss: 0.6919
Epoch 2/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5368 - loss: 0.6905 - val_accuracy: 0.5966 - val_loss: 0.6753
Epoch 3/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5263 - loss: 0.6900 - val_accuracy: 0.6471 - val_loss: 0.6790
Epoch 4/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5411 - loss: 0.6838 - val_accuracy: 0.6134 - val_loss: 0.6735
Epoch 5/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5747 - loss: 0.6570 - val_accuracy: 0.4454 - val_loss: 0.7134
Epoch 6/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5832 - loss: 0.6588 - val_accuracy: 0.5966 - val_loss: 0.6477
Epoch 7/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5263 - loss: 0.6780 - val_accuracy: 0.4454 - val_loss: 0.6939
Epoch 8/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5811 - loss: 0.6793 - val_accuracy: 0.6134 - v

✅ Modelo re-entrenado para YOLO listo.


In [19]:
import cv2
from ultralytics import YOLO
import numpy as np
from tensorflow.keras.models import load_model
import time

# 1. CARGA DE MODELOS
model_ia = load_model('modelo_squat_yolo.h5')
yolo_pose = YOLO('yolov8n-pose.pt')

# 2. VÍDEO
cap = cv2.VideoCapture(r"C:\Users\Anxo\Downloads\455kg squat FIGHT Jesus Olivares.mp4")

# 3. LÓGICA DE CONTROL AVANZADA
secuencia = []
contador_reps = 0
estado = "ESPERANDO" 
mejor_confianza_valida = 0
y_hombros_inicial = None
frames_en_bajada = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    frame = cv2.resize(frame, (800, 600))
    results = yolo_pose.predict(frame, conf=0.3, verbose=False)
    
    if results[0].keypoints and len(results[0].keypoints.data) > 0:
        kp_data = results[0].keypoints.data[0].cpu().numpy()
        hombro_y = (kp_data[5][1] + kp_data[6][1]) / 2
        
        if y_hombros_inicial is None:
            y_hombros_inicial = hombro_y
            continue

        # Guardar secuencia
        kp_flat = kp_data.flatten()
        if len(kp_flat) == 51:
            secuencia.append(kp_flat)
        if len(secuencia) > 30: secuencia.pop(0)

        # --- LÓGICA DE DECISIÓN ROBUSTA ---
        distancia_bajada = hombro_y - y_hombros_inicial

        # A) Detección de inicio de bajada
        if distancia_bajada > 40 and estado == "ESPERANDO":
            estado = "BAJANDO"
            mejor_confianza_valida = 0
            frames_en_bajada = 0

        # B) Durante la bajada, acumulamos la opinión de la IA
        if estado == "BAJANDO":
            frames_en_bajada += 1
            if len(secuencia) == 30:
                res = model_ia.predict(np.expand_dims(secuencia, axis=0), verbose=0)[0]
                # Guardamos la probabilidad de que sea VALIDA (clase 1)
                prob_valida = res[1]
                if prob_valida > mejor_confianza_valida:
                    mejor_confianza_valida = prob_valida

            # C) Detección de subida (Jesús recupera posición)
            if distancia_bajada < 30 and frames_en_bajada > 20:
                # Si en algún momento de la bajada la IA vio una sentadilla válida con > 70%
                if mejor_confianza_valida > 0.70:
                    contador_reps += 1
                    print(f"✅ Repetición detectada con éxito. Confianza: {mejor_confianza_valida}")
                estado = "ESPERANDO"

    # --- HUD VISUAL ---
    frame_final = results[0].plot()
    color = (0, 255, 0) if estado == "BAJANDO" else (255, 255, 255)
    cv2.putText(frame_final, f"ESTADO: {estado}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    cv2.putText(frame_final, f"REPS: {contador_reps}", (600, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
    
    cv2.imshow('Analizador Jesus Olivares - Final Fix', frame_final)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

✅ Repetición detectada con éxito. Confianza: 1.0


In [1]:
import cv2
from ultralytics import YOLO
import numpy as np
from tensorflow.keras.models import load_model

model_ia = load_model('modelo_squat_yolo.h5')
yolo_pose = YOLO('yolov8n-pose.pt')
cap = cv2.VideoCapture(r"C:\Users\Anxo\Downloads\455kg squat FIGHT Jesus Olivares.mp4")

secuencia = []
contador_reps = 0
estado = "ESPERANDO"
mejor_confianza_valida = 0
y_hombros_inicial = None
luces_color = (100, 100, 100) # Gris inicial (apagadas)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame = cv2.resize(frame, (800, 600))
    results = yolo_pose.predict(frame, conf=0.3, verbose=False)
    
    if results[0].keypoints and len(results[0].keypoints.data) > 0:
        kp_data = results[0].keypoints.data[0].cpu().numpy()
        hombro_y = (kp_data[5][1] + kp_data[6][1]) / 2
        if y_hombros_inicial is None: y_hombros_inicial = hombro_y; continue

        kp_flat = kp_data.flatten()
        if len(kp_flat) == 51: secuencia.append(kp_flat)
        if len(secuencia) > 30: secuencia.pop(0)

        distancia_bajada = hombro_y - y_hombros_inicial

        if distancia_bajada > 40 and estado == "ESPERANDO":
            estado = "BAJANDO"
            mejor_confianza_valida = 0
            luces_color = (100, 100, 100) # Reset luces al bajar

        if estado == "BAJANDO" and len(secuencia) == 30:
            res = model_ia.predict(np.expand_dims(secuencia, axis=0), verbose=0)[0]
            if res[1] > mejor_confianza_valida: mejor_confianza_valida = res[1]

        if distancia_bajada < 30 and estado == "BAJANDO":
            if mejor_confianza_valida > 0.70:
                contador_reps += 1
                luces_color = (255, 255, 255) # BLANCO (Válida)
            else:
                luces_color = (0, 0, 255) # ROJO (Nula)
            estado = "ESPERANDO"

    # --- DIBUJAR SEMÁFORO DE JUECES ---
    frame_final = results[0].plot()
    # Dibujamos 3 círculos (Luces de Powerlifting)
    for i in range(3):
        cv2.circle(frame_final, (600 + (i*50), 40), 20, luces_color, -1)
        cv2.circle(frame_final, (600 + (i*50), 40), 20, (0, 0, 0), 2) # Borde

    cv2.putText(frame_final, f"REPS: {contador_reps}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
    cv2.imshow('Juez Virtual de Powerlifting', frame_final)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [1]:
import cv2
from ultralytics import YOLO
import numpy as np
from tensorflow.keras.models import load_model

# 1. CARGA DE MODELOS (Usamos el mismo cerebro que entrenamos)
model_ia = load_model('modelo_squat_yolo.h5')
yolo_pose = YOLO('yolov8n-pose.pt')

# 2. ACTIVAR WEBCAM (El 0 suele ser la cámara integrada)
cap = cv2.VideoCapture(0)

# 3. VARIABLES DE CONTROL
secuencia = []
contador_reps = 0
estado = "ESPERANDO"
mejor_confianza_valida = 0
y_hombros_inicial = None
luces_color = (100, 100, 100) # Gris (Apagadas)

print("🎥 Cámara activada. Ponte de pie frente a la cámara para calibrar...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    # Espejo (opcional, para que te veas más natural)
    frame = cv2.flip(frame, 1)
    frame = cv2.resize(frame, (840, 640))
    
    # Detección de Pose
    results = yolo_pose.predict(frame, conf=0.5, verbose=False)
    
    if results[0].keypoints and len(results[0].keypoints.data) > 0:
        kp_data = results[0].keypoints.data[0].cpu().numpy()
        
        # Punto medio de hombros para el trigger
        hombro_y = (kp_data[5][1] + kp_data[6][1]) / 2
        
        # Calibración automática al inicio
        if y_hombros_inicial is None:
            y_hombros_inicial = hombro_y
            continue

        # Guardar datos para la IA
        kp_flat = kp_data.flatten()
        if len(kp_flat) == 51:
            secuencia.append(kp_flat)
        if len(secuencia) > 30: secuencia.pop(0)

        # --- LÓGICA DE JUEZ EN TIEMPO REAL ---
        distancia_bajada = hombro_y - y_hombros_inicial

        # Detectar cuando empiezas a bajar
        if distancia_bajada > 50 and estado == "ESPERANDO":
            estado = "BAJANDO"
            mejor_confianza_valida = 0
            luces_color = (100, 100, 100)

        # Analizar técnica durante la bajada
        if estado == "BAJANDO" and len(secuencia) == 30:
            res = model_ia.predict(np.expand_dims(secuencia, axis=0), verbose=0)[0]
            if res[1] > mejor_confianza_valida:
                mejor_confianza_valida = res[1]

        # Detectar cuando vuelves a estar arriba
        if distancia_bajada < 35 and estado == "BAJANDO":
            if mejor_confianza_valida > 0.65: # Un pelín más flexible para webcam
                contador_reps += 1
                luces_color = (255, 255, 255) # BLANCO (Válida)
            else:
                luces_color = (0, 0, 255) # ROJO (Nula)
            estado = "ESPERANDO"

    # --- DIBUJAR INTERFAZ DE COMPETICIÓN ---
    frame_final = results[0].plot()
    
    # Dibujar las 3 luces de Powerlifting
    for i in range(3):
        cv2.circle(frame_final, (700 + (i*45), 50), 18, luces_color, -1)
        cv2.circle(frame_final, (700 + (i*45), 50), 18, (0,0,0), 2)

    # Info en pantalla
    cv2.rectangle(frame_final, (10, 10), (250, 90), (0,0,0), -1)
    cv2.putText(frame_final, f"REPS: {contador_reps}", (30, 45), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
    cv2.putText(frame_final, f"ESTADO: {estado}", (30, 75), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 1)

    cv2.imshow('Juez IA - Live Mode', frame_final)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

🎥 Cámara activada. Ponte de pie frente a la cámara para calibrar...


In [2]:
import cv2
from ultralytics import YOLO
import numpy as np
from tensorflow.keras.models import load_model

# 1. CARGA
model_ia = load_model('modelo_squat_yolo.h5')
yolo_pose = YOLO('yolov8n-pose.pt')
cap = cv2.VideoCapture(r"C:\Users\Anxo\Downloads\455kg squat FIGHT Jesus Olivares.mp4")

secuencia = []
mejor_confianza = 0
juego_estado = "JUGANDO"
voto_usuario = None
veredicto_ia = None

print("🎮 BIENVENIDO AL JUEGO DEL JUEZ")
print("Observa el levantamiento... Al finalizar, pulsa 'v' para VALIDA o 'n' para NULA.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    frame = cv2.resize(frame, (800, 600))
    results = yolo_pose.predict(frame, conf=0.3, verbose=False)
    
    if results[0].keypoints and len(results[0].keypoints.data) > 0:
        kp = results[0].keypoints.data[0].cpu().numpy().flatten()
        if len(kp) == 51:
            secuencia.append(kp)
            if len(secuencia) > 30: secuencia.pop(0)
            
            # La IA va analizando en segundo plano
            if len(secuencia) == 30:
                res = model_ia.predict(np.expand_dims(secuencia, axis=0), verbose=0)[0]
                if res[1] > mejor_confianza: mejor_confianza = res[1]

    # Dibujamos el esqueleto
    frame_final = results[0].plot()
    
    # Interfaz del juego
    cv2.rectangle(frame_final, (0, 0), (800, 60), (50, 50, 50), -1)
    cv2.putText(frame_final, "¿ES VALIDA O NULA? (Mira el video)", (150, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    cv2.imshow('Juego: Juez de Powerlifting', frame_final)
    cv2.waitKey(20) # Velocidad de reproducción

# --- FASE DE VOTACIÓN ---
print("\n--- EL VÍDEO HA TERMINADO ---")
while voto_usuario is None:
    tecla = cv2.waitKey(0) & 0xFF
    if tecla == ord('v'): voto_usuario = "VALIDA"
    elif tecla == ord('n'): voto_usuario = "NULA"

# La IA da su veredicto final basado en lo que vio
veredicto_ia = "VALIDA" if mejor_confianza > 0.70 else "NULA"

# --- PANTALLA DE RESULTADOS ---
resultado_img = np.zeros((600, 800, 3), dtype=np.uint8)
color_resultado = (0, 255, 0) if voto_usuario == veredicto_ia else (0, 0, 255)
mensaje = "¡HAS ACERTADO!" if voto_usuario == veredicto_ia else "¡HAS FALLADO!"

cv2.putText(resultado_img, mensaje, (200, 150), cv2.FONT_HERSHEY_SIMPLEX, 2, color_resultado, 4)
cv2.putText(resultado_img, f"Tu voto: {voto_usuario}", (100, 300), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
cv2.putText(resultado_img, f"Veredicto IA: {veredicto_ia}", (100, 400), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
cv2.putText(resultado_img, f"Confianza IA: {mejor_confianza*100:.1f}%", (100, 500), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (200, 200, 200), 1)

cv2.imshow('Juego: Juez de Powerlifting', resultado_img)
cv2.waitKey(0)
cap.release()
cv2.destroyAllWindows()

🎮 BIENVENIDO AL JUEGO DEL JUEZ
Observa el levantamiento... Al finalizar, pulsa 'v' para VALIDA o 'n' para NULA.

--- EL VÍDEO HA TERMINADO ---


In [7]:
import os
import numpy as np
import cv2
import random
from tensorflow.keras.models import load_model

# 1. CARGA DEL MODELO (Asegúrate de tener el archivo .h5 en la misma carpeta)
try:
    model_ia = load_model('modelo_squat_yolo.h5')
    print("✅ Modelo IA cargado correctamente.")
except:
    print("❌ Error: No se encuentra 'modelo_squat_yolo.h5'. Ejecuta primero la celda de entrenamiento.")
    raise

# 2. RUTA RAÍZ (Ajusta si es necesario)
ruta_raiz = r"C:\Users\Anxo\Desktop\Proyecto final 4\archive (3)\Squat_Data\Squat_Data"

# 3. CONEXIONES DE MEDIAPIPE (Stickman)
# Definimos qué puntos se conectan entre sí para formar el esqueleto
CONEXIONES_STICKMAN = [
    # Torso y Cabeza (Rojo)
    ((11, 12), (255, 0, 0)), ((11, 23), (255, 0, 0)), ((12, 24), (255, 0, 0)), ((23, 24), (255, 0, 0)),
    ((8, 6), (255, 0, 0)), ((6, 5), (255, 0, 0)), ((5, 4), (255, 0, 0)), ((4, 0), (255, 0, 0)),
    ((0, 1), (255, 0, 0)), ((1, 2), (255, 0, 0)), ((2, 3), (255, 0, 0)), ((3, 7), (255, 0, 0)),
    # Brazos (Verde)
    ((11, 13), (0, 255, 0)), ((13, 15), (0, 255, 0)), ((15, 17), (0, 255, 0)), ((15, 19), (0, 255, 0)), ((15, 21), (0, 255, 0)),
    ((12, 14), (0, 255, 0)), ((14, 16), (0, 255, 0)), ((16, 18), (0, 255, 0)), ((16, 20), (0, 255, 0)), ((16, 22), (0, 255, 0)),
    # Piernas (Azul)
    ((23, 25), (0, 0, 255)), ((25, 27), (0, 0, 255)), ((27, 29), (0, 0, 255)), ((27, 31), (0, 0, 255)),
    ((24, 26), (0, 0, 255)), ((26, 28), (0, 0, 255)), ((28, 30), (0, 0, 255)), ((28, 32), (0, 0, 255))
]

def dibujar_stickman(puntos_132):
    """ Dibuja el stickman articulado en un lienzo negro """
    lienzo = np.zeros((600, 800, 3), dtype=np.uint8)
    puntos_codificados = []
    
    # 1. Extraer y escalar puntos
    for i in range(33):
        x = int(puntos_132[i*4] * 800)
        y = int(puntos_132[i*4 + 1] * 600)
        puntos_codificados.append((x, y))
        # Dibujar punto
        if 0 <= x < 800 and 0 <= y < 600:
            cv2.circle(lienzo, (x, y), 3, (255, 255, 255), -1)

    # 2. Dibujar líneas de conexión
    for conexion, color in CONEXIONES_STICKMAN:
        p1_idx, p2_idx = conexion
        p1 = puntos_codificados[p1_idx]
        p2 = puntos_codificados[p2_idx]
        
        # Validar que los puntos estén dentro del lienzo
        if (0 <= p1[0] < 800 and 0 <= p1[1] < 600) and \
           (0 <= p2[0] < 800 and 0 <= p2[1] < 600):
            cv2.line(lienzo, p1, p2, color, 2)
            
    return lienzo

def jugar_ronda_visual():
    """ Ejecuta una ronda del minijuego """
    # Elección aleatoria de clase y carpeta
    clase_real = random.choice([0, 1])
    nombre_clase = "Valid" if clase_real == 1 else "Invalid"
    ruta_clase = os.path.join(ruta_raiz, nombre_clase)
    
    carpetas = [d for d in os.listdir(ruta_clase) if os.path.isdir(os.path.join(ruta_clase, d))]
    video_id = random.choice(carpetas)
    ruta_video = os.path.join(ruta_clase, video_id)
    archivos = sorted([f for f in os.listdir(ruta_video) if f.endswith('.npy')])
    
    if len(archivos) < 30: return jugar_ronda_visual() # Reintentar si es corto

    secuencia_ia = []
    indices_yolo = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
    
    # --- REPRODUCCIÓN DEL STICKMAN ---
    print(f"🎬 Analizando movimiento en: {nombre_clase}/{video_id}...")
    for i in range(min(len(archivos), 70)): # Mostramos hasta 70 frames para ver todo el rango
        data = np.load(os.path.join(ruta_video, archivos[i]))
        
        # Generar stickman coloreado
        img_stickman = dibujar_stickman(data)
        cv2.putText(img_stickman, "OBSERVA LA PROFUNDIDAD", (150, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        cv2.imshow('Minijuego: Stickman Squat Judge', img_stickman)
        cv2.waitKey(40) # Velocidad de reproducción (40ms = 25fps aprox)
        
        # Guardamos secuencia para la IA (solo los primeros 30 frames)
        if i < 30:
            puntos_51 = []
            for idx in indices_yolo:
                puntos_51.extend(data[idx*4 : idx*4+3])
            secuencia_ia.append(puntos_51)

    # --- FASE DE VOTACIÓN ---
    print("\n❓ ¿Es VALIDA (pulsa 'v') o NULA (pulsa 'n')?")
    voto = None
    while voto not in ['v', 'n']:
        tecla = cv2.waitKey(0) & 0xFF
        if tecla == ord('v'): voto = 'v'
        elif tecla == ord('n'): voto = 'n'
    
    voto_num = 1 if voto == 'v' else 0
    
    # Predicción IA
    pred = model_ia.predict(np.expand_dims(secuencia_ia, axis=0), verbose=0)[0]
    ia_decide = np.argmax(pred)
    ia_conf = pred[ia_decide] * 100
    
    # --- PANTALLA DE RESULTADOS ---
    resultado_img = np.zeros((600, 800, 3), dtype=np.uint8)
    color_voto = (0, 255, 0) if voto_num == clase_real else (0, 0, 255)
    color_ia = (0, 255, 0) if ia_decide == clase_real else (0, 0, 255)
    
    cv2.putText(resultado_img, f"TU VOTO: {voto.upper()}", (100, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255,255,255), 3)
    cv2.putText(resultado_img, f"REAL DE LA DB: {nombre_clase.upper()}", (100, 250), cv2.FONT_HERSHEY_SIMPLEX, 1.5, color_voto, 3)
    cv2.putText(resultado_img, f"IA DICE: {'VALIDA' if ia_decide == 1 else 'NULA'}", (100, 380), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (200,200,200), 2)
    cv2.putText(resultado_img, f"Confianza IA: {ia_conf:.1f}%", (100, 430), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (150,150,150), 1)
    
    # Mensaje final
    msj_final = "¡COINCIDES CON LA DB!" if voto_num == clase_real else "¡LA DB DICE LO CONTRARIO!"
    cv2.putText(resultado_img, msj_final, (100, 520), cv2.FONT_HERSHEY_SIMPLEX, 1, color_voto, 2)
    cv2.putText(resultado_img, "Pulsa cualquier tecla para otra ronda ('q' para salir)", (120, 570), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)
    
    cv2.imshow('Minijuego: Stickman Squat Judge', resultado_img)
    
    # Esperar tecla para continuar o salir
    final_tecla = cv2.waitKey(0) & 0xFF
    if final_tecla == ord('q'):
        return False
    return True

# --- BUCLE PRINCIPAL DEL JUEGO ---
print("🎮 BIENVENIDO AL JUEGO 'STICKMAN SQUAT JUDGE'")
print("Instrucciones: Mira el movimiento del muñeco articulado.")
print("Cuando se detenga, pulsa 'v' para Válida o 'n' para Nula.")
print("Pulsa 'q' en la pantalla de resultados para salir.")

try:
    jugando = True
    while jugando:
        jugando = jugar_ronda_visual()
except KeyboardInterrupt:
    print("\nJuego interrumpido por el usuario.")
finally:
    cv2.destroyAllWindows()
    print("👋 ¡Gracias por jugar! El juez de hierro se despide.")

✅ Modelo IA cargado correctamente.
🎮 BIENVENIDO AL JUEGO 'STICKMAN SQUAT JUDGE'
Instrucciones: Mira el movimiento del muñeco articulado.
Cuando se detenga, pulsa 'v' para Válida o 'n' para Nula.
Pulsa 'q' en la pantalla de resultados para salir.
🎬 Analizando movimiento en: Valid/72...

❓ ¿Es VALIDA (pulsa 'v') o NULA (pulsa 'n')?
🎬 Analizando movimiento en: Invalid/65...

❓ ¿Es VALIDA (pulsa 'v') o NULA (pulsa 'n')?
🎬 Analizando movimiento en: Valid/18...

❓ ¿Es VALIDA (pulsa 'v') o NULA (pulsa 'n')?
🎬 Analizando movimiento en: Valid/87...

❓ ¿Es VALIDA (pulsa 'v') o NULA (pulsa 'n')?
🎬 Analizando movimiento en: Valid/27...

❓ ¿Es VALIDA (pulsa 'v') o NULA (pulsa 'n')?
🎬 Analizando movimiento en: Invalid/95...

❓ ¿Es VALIDA (pulsa 'v') o NULA (pulsa 'n')?
🎬 Analizando movimiento en: Invalid/43...

❓ ¿Es VALIDA (pulsa 'v') o NULA (pulsa 'n')?

Juego interrumpido por el usuario.
👋 ¡Gracias por jugar! El juez de hierro se despide.


In [9]:
import cv2
from ultralytics import YOLO
import numpy as np
from tensorflow.keras.models import load_model
import time

# 1. CARGA DE MODELOS
model_ia = load_model('modelo_squat_yolo.h5')
yolo_pose = YOLO('yolov8n-pose.pt')

# 2. VÍDEO (Cambia a 0 para Webcam)
cap = cv2.VideoCapture(r"C:\Users\Anxo\Downloads\455kg squat FIGHT Jesus Olivares.mp4")

# 3. VARIABLES DE CONTROL Y BIOMECÁNICA
secuencia = []
contador_reps = 0
estado = "ESPERANDO"
mejor_confianza_valida = 0
y_hombros_inicial = None
max_inclinacion = 0
consejo_coach = "SISTEMA LISTO. ESPERANDO DESCENSO..."
luces_color = (100, 100, 100)

def calcular_angulo_torso(puntos):
    """Calcula la inclinación del torso respecto a la vertical"""
    # Hombro (5 o 6) y Cadera (11 o 12)
    hombro = puntos[5]
    cadera = puntos[11]
    
    dy = cadera[1] - hombro[1]
    dx = cadera[0] - hombro[0]
    
    # Ángulo en grados
    angulo = np.abs(np.degrees(np.arctan2(dx, dy)))
    return angulo

print("🚀 Coach Inteligente Activado...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    frame = cv2.resize(frame, (800, 600))
    results = yolo_pose.predict(frame, conf=0.3, verbose=False)
    
    if results[0].keypoints and len(results[0].keypoints.data) > 0:
        kp_data = results[0].keypoints.data[0].cpu().numpy()
        hombro_y = (kp_data[5][1] + kp_data[6][1]) / 2
        
        if y_hombros_inicial is None:
            y_hombros_inicial = hombro_y
            continue

        # Guardar secuencia para la IA
        kp_flat = kp_data.flatten()
        if len(kp_flat) == 51:
            secuencia.append(kp_flat)
        if len(secuencia) > 30: secuencia.pop(0)

        # --- LÓGICA DE COACHING ---
        distancia_bajada = hombro_y - y_hombros_inicial
        angulo_actual = calcular_angulo_torso(kp_data)

        if distancia_bajada > 40 and estado == "ESPERANDO":
            estado = "BAJANDO"
            mejor_confianza_valida = 0
            max_inclinacion = 0
            consejo_coach = "BAJANDO... MANTEN EL PECHO ALTO"
            luces_color = (100, 100, 100)

        if estado == "BAJANDO":
            # Registrar la peor inclinación durante la bajada
            if angulo_actual > max_inclinacion:
                max_inclinacion = angulo_actual
            
            if len(secuencia) == 30:
                res = model_ia.predict(np.expand_dims(secuencia, axis=0), verbose=0)[0]
                if res[1] > mejor_confianza_valida:
                    mejor_confianza_valida = res[1]

        # Al subir y terminar la rep
        if distancia_bajada < 30 and estado == "BAJANDO":
            # Veredicto de profundidad (IA)
            if mejor_confianza_valida > 0.70:
                contador_reps += 1
                luces_color = (255, 255, 255)
                # Feedback de técnica
                if max_inclinacion > 35:
                    consejo_coach = f"VALIDA. Cuidado: Torso inclinado ({int(max_inclinacion)}°). Fortalece Lumbar."
                else:
                    consejo_coach = "¡PERFECTA! Buena verticalidad y profundidad."
            else:
                luces_color = (0, 0, 255)
                consejo_coach = "NULA. Falta profundidad. Mejora movilidad de tobillo."
            
            estado = "ESPERANDO"

    # --- RENDERIZADO DEL COACH HUD ---
    frame_final = results[0].plot()
    
    # Caja de consejos inferior
    cv2.rectangle(frame_final, (0, 530), (800, 600), (0, 0, 0), -1)
    cv2.putText(frame_final, consejo_coach, (20, 570), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    
    # Luces de Juez
    for i in range(3):
        cv2.circle(frame_final, (650 + (i*45), 45), 18, luces_color, -1)
        cv2.circle(frame_final, (650 + (i*45), 45), 18, (255, 255, 255), 1)

    # Marcador superior
    cv2.rectangle(frame_final, (0, 0), (200, 60), (0, 0, 0), -1)
    cv2.putText(frame_final, f"REPS: {contador_reps}", (20, 45), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    cv2.imshow('AI SQUAT COACH - Jesus Olivares Edition', frame_final)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

🚀 Coach Inteligente Activado...


In [11]:
import streamlit as st
import cv2
from ultralytics import YOLO
import numpy as np
from tensorflow.keras.models import load_model

# Configuración de la página
st.set_page_config(page_title="Squat AI Pro Coach", layout="wide")

# 1. CARGA DE MODELOS (Usamos cache para que no tarde cada vez)
@st.cache_resource
def load_models():
    model_ia = load_model('modelo_squat_yolo.h5')
    yolo_pose = YOLO('yolov8n-pose.pt')
    return model_ia, yolo_pose

model_ia, yolo_pose = load_models()

# --- BARRA LATERAL ---
st.sidebar.title("🎮 Panel de Control")
modo = st.sidebar.selectbox("Selecciona un modo:", 
                           ["Juez de Vídeo", "Coach en Tiempo Real", "Minijuego de Entrenamiento"])

st.sidebar.divider()
st.sidebar.info("Proyecto Final: Analizador Biomecánico de Sentadillas")

# --- MODO 1: JUEZ DE VÍDEO ---
if modo == "Juez de Vídeo":
    st.title("⚖️ Juez de Competición")
    video_file = st.file_uploader("Sube un vídeo de tu levantamiento", type=['mp4', 'mov', 'avi'])
    
    if video_file:
        # Guardar temporalmente el vídeo
        with open("temp_video.mp4", "wb") as f:
            f.write(video_file.read())
            
        cap = cv2.VideoCapture("temp_video.mp4")
        st_frame = st.empty() # Espacio para el vídeo procesado
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            # --- Aquí iría tu lógica de procesamiento que ya tenemos ---
            # (YOLO + LSTM + Luces)
            
            # Mostrar en la interfaz de Streamlit
            st_frame.image(frame, channels="BGR", use_container_width=True)
        cap.release()

# --- MODO 2: COACH EN TIEMPO REAL ---
elif modo == "Coach en Tiempo Real":
    st.title("🤖 Coach Inteligente (Webcam)")
    st.warning("Asegúrate de tener buena iluminación y encuadre de cuerpo completo.")
    
    run_webcam = st.checkbox('Activar Cámara')
    st_webcam = st.empty()
    col1, col2 = st.columns(2)
    with col1:
        reps_stat = st.metric("Repeticiones", "0")
    with col2:
        coach_msg = st.chat_message("assistant")
        coach_msg.write("Esperando a que empieces...")

    if run_webcam:
        cap = cv2.VideoCapture(0)
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            # Lógica de coaching y actualización de st_webcam, reps_stat y coach_msg
            st_webcam.image(frame, channels="BGR")
        cap.release()

# --- MODO 3: MINIJUEGO ---
elif modo == "Minijuego de Entrenamiento":
    st.title("🕹️ Stickman Challenge")
    st.write("¿Eres capaz de superar a la IA juzgando profundidad?")
    # Aquí pegamos la lógica del Stickman que ya funciona

2026-03-23 02:20:05.391 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-23 02:20:05.393 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-23 02:20:05.492 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-23 02:20:05.693 
  command:

    streamlit run c:\Users\Anxo\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-23 02:20:05.694 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-23 02:20:05.694 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-23 02: